In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import cv2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential , load_model
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Dropout
from sklearn.model_selection import train_test_split
from collections import deque


In [ ]:
train_df = pd.read_csv('sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test.csv')

In [ ]:
#so in train csv we have to remove lables and keep only image pixel values
X_train = train_df.iloc[:,1:].values.reshape(-1,28,28,1)/255.0

#iloc[:,1:] removes lables in the first coloumn
#reshaping each row into 28x28 grayscale image
#/255.0 normalizes pixel values for better training 
y_train =to_categorical(train_df['label'])
#selects column of lables and convrts lablels to one hot encoding

In [ ]:
#Same for test
X_test = test_df.iloc[:,1:].values.reshape(-1,28,28,1)/255.0
y_test = to_categorical(test_df['label'])


In [ ]:
#Building CNN model
model = Sequential([
    Conv2D(32,(3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D(2,2),
    
    Conv2D(64,(3,3), activation = 'relu'),
    MaxPooling2D(2,2),
    
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(25,activation = 'softmax')
])
model.compile(optimizer = 'adam',loss ='categorical_crossentropy', metrics = ['accuracy'])
model.fit(X_train, y_train, epochs= 100, validation_data=(X_test,y_test))
model.save('asl_model.h5')

Epoch 1/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - accuracy: 0.3527 - loss: 2.1356 - val_accuracy: 0.8383 - val_loss: 0.5039
Epoch 2/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.8546 - loss: 0.4263 - val_accuracy: 0.9168 - val_loss: 0.2633
Epoch 3/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.9333 - loss: 0.1943 - val_accuracy: 0.9137 - val_loss: 0.2748
Epoch 4/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.9630 - loss: 0.1146 - val_accuracy: 0.9353 - val_loss: 0.2606
Epoch 5/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.9703 - loss: 0.0885 - val_accuracy: 0.9398 - val_loss: 0.2307
Epoch 6/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.9781 - loss: 0.0669 - val_accuracy: 0.9289 - val_loss: 0.2173
Epoch 7/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step - accuracy: 0.9813 - loss: 0.0543 - val_accuracy: 0.9290 - val_loss: 0.2239
Epoch 8/100
858/858 ━━━━━━━━━━━━━━━━━━━━ 46s 32ms/step - accuracy: 0.9811 - loss: 0

In [ ]:

#real time prediction using webcam
model = load_model('asl_model.h5')
classes =[chr(i) for i in range(65,91) if i!=74]

cap=cv2.VideoCapture(0)
# Create a buffer of last 20 predictions
prediction_buffer = deque(maxlen=20)
while True:
    ret,frame = cap.read()
    frame = cv2.flip(frame,1)
    
    #Define ROI
    
    x1,y1,x2,y2=100,100,324,324
    roi = frame[y1:y2,x1:x2]
    img = cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img,(28,28))
    img = img.reshape(1,28,28,1) / 255.0
    
    prediction = model.predict(img)
    class_id = np.argmax(prediction)
    letter = classes[class_id]
    # Add prediction to buffer
    prediction_buffer.append(letter)
    


    most_common = max(set(prediction_buffer), key=prediction_buffer.count)
    cv2.rectangle(frame, (x1,y1),(x2,y2),(0,255,0),2)
    cv2.putText(frame,f'Prediction: {letter}',(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,0),2)
    cv2.imshow("Sign Language Translator",frame)
    
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/s

: 